# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

All entities (record sets, fields, etc.) are referenced by their `@id`, following best practices for Croissant interoperability.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the general metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Citation: {getattr(meta, 'citeAs', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(meta, 'keywords', [])}\n")
print(f"Record sets in schema: {len(meta.record_sets)}")

## 2. Data Overview
Enumerate available record sets, listing their `@id`, labels, and available fields. All references use the entity `@id` per Croissant/FAIR best practice.

In [ ]:
# List all record sets and their fields (by @id)
for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs.label} (id: {rs.id})")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            ctype = getattr(field, 'data_type', 'N/A')
            print(f"   - Field: {field.label} (@id: {field.id}, type: {ctype})")
    print()

## 3. Data Extraction
Load data from a record set into a DataFrame. By default, we use the primary `proteins` record set: `- `@id`: `cr:recordSet/proteins`

Replace `record_set_id` and field `@id`s below as needed if exploring other record sets.

In [ ]:
# List all record set @ids for reference
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Available record sets:", record_set_ids)

# Choose primary record set (by @id)
record_set_id = 'cr:recordSet/proteins'

# Extract records for each record set into DataFrame
dfs = {}
for rid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rid))
        dfs[rid] = pd.DataFrame(records)
        print(f"Loaded '{rid}', shape: {dfs[rid].shape}")
    except Exception as e:
        print(f"Could not load '{rid}':", e)

# Explore columns of main record set
if record_set_id in dfs:
    print(f"\nColumns for {record_set_id}:")
    print(dfs[record_set_id].columns.tolist())
    display(dfs[record_set_id].head())
else:
    print(f"{record_set_id} not found.")

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping using field `@id`s from the schema. Example: process the `cr:field/mw` (molecular weight) field if available.

In [ ]:
# Choose fields by @id for numeric and grouping analysis
numeric_field_id = 'cr:field/mw'  # Example: field for molecular weight (Da)
group_field_id = 'cr:field/peptide_count_unique'  # Example: unique peptide count

df = dfs.get(record_set_id)
if df is not None and numeric_field_id in df.columns:
    # Only keep numeric values (could be str if loaded so try conversion)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 50000
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print("\nNormalized field:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by peptide count (if available)
    if group_field_id in filtered_df.columns:
        filtered_df[group_field_id] = pd.to_numeric(filtered_df[group_field_id], errors='coerce')
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(f"\nAverage MW grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Column '{numeric_field_id}' not found in {record_set_id}.")

## 5. Visualization
Visualize the normalized molecular weight field and its distribution, and the relationship with unique peptide count (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=50, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (Molecular Weight)")
    plt.xlabel("Molecular Weight (Da)")
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8,6))
        sns.scatterplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} vs {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library:
- Loaded the dataset via its Croissant schema using its URL
- Examined record sets and their field structure using their `@id`s
- Loaded the `cr:recordSet/proteins` record set and performed EDA on molecular weight (`cr:field/mw`)
- Visualized its distribution and explored its relation to peptide count

Further analyses can explore additional record sets or relationships between fields, always referencing fields by their Croissant `@id` for precise, reproducible workflows.